# Plan B / Option E: Learning Rate Comparison with SGD

This notebook trains a TF-IDF + SGD classifier on the same synthetic financial sentiment data,
running **all three learning rate presets simultaneously** so you can see them on the same plot.

It teaches the same Unit 6 optimisation story as the main notebook:
- **loss function** — measuring how wrong we are
- **stochastic gradient descent (SGD)** — taking downhill steps using mini-batches
- **learning rate** — controlling step size
- **generalisation gap** — train vs validation performance

**Why this is useful even if you ran the main notebook:**
Each group in the main experiment only ran one preset. This shows all three on the same axes
so you can compare curve shapes directly. The patterns are also more visible here because the
data includes intentional label noise (25% of training labels are randomly corrupted) —
this makes the three failure modes dramatically distinct.


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
import matplotlib.pyplot as plt

np.random.seed(42)


In [ ]:
# Load local synthetic dataset (offline-friendly)
# Local dataset path (robust)
csv_candidates = [
    os.path.join("lab", "data", "synth_financial_sentiment.csv"),
    os.path.join("data", "synth_financial_sentiment.csv"),
]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(f"Could not find synth dataset. Tried: {csv_candidates}")

# Optional: add label noise to make learning-rate effects more visible (set to 0.0 for clean data)
NOISE_FRACTION = 0.25
df = pd.read_csv(csv_path)

# Inject label noise (optional)
if NOISE_FRACTION and NOISE_FRACTION > 0:
    rng = np.random.default_rng(42)
    flip_idx = rng.choice(df.index, size=int(NOISE_FRACTION * len(df)), replace=False)
    labels = ["negative", "neutral", "positive"]
    for idx in flip_idx:
        current = df.at[idx, "label"]
        other = [l for l in labels if l != current]
        df.at[idx, "label"] = rng.choice(other)


label_order = ["negative", "neutral", "positive"]
label2id = {name: i for i, name in enumerate(label_order)}
df["y"] = df["label"].map(label2id)

X_train, X_val, y_train, y_val = train_test_split(
    df["sentence"].to_numpy(), df["y"].to_numpy(),
    test_size=0.2, random_state=42, stratify=df["y"].to_numpy()
)

vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

X_train_vec.shape, X_val_vec.shape


## What these three presets show

We run three experiments in sequence with the same architecture and data, varying only `eta0` (the learning rate):

| Preset | eta0 | What to expect |
|--------|------|----------------|
| `TOO_LOW` | 0.0001 | Steps so tiny the weights barely move — model stays near random baseline |
| `JUST_RIGHT` | 0.01 | Steady improvement — finds the signal despite the noise |
| `TOO_HIGH` | 5.0 | Overshoots so badly it memorises the noise — validation loss *increases* each epoch |

**The noise matters:** 25% of training labels are randomly corrupted. This creates a floor
on achievable accuracy (~75%) and makes the high learning rate failure mode visible as
*active divergence* rather than mere instability.


In [ ]:
ETA_MAP = {
    "TOO_LOW": 1e-4,
    "JUST_RIGHT": 1e-2,
    "TOO_HIGH": 5.0,
}

EPOCHS = 10


In [ ]:
def run_sgd_experiment(eta0: float):
    clf = SGDClassifier(
        loss="log_loss",
        learning_rate="constant",
        eta0=eta0,
        alpha=1e-4,
        random_state=42,
    )
    classes = np.array([0,1,2])
    train_losses=[]
    val_losses=[]
    val_acc=[]
    val_f1=[]

    # initialise
    clf.partial_fit(X_train_vec, y_train, classes=classes)

    for _ in range(EPOCHS):
        clf.partial_fit(X_train_vec, y_train)
        # predict_proba exists for log_loss
        train_prob = clf.predict_proba(X_train_vec)
        val_prob = clf.predict_proba(X_val_vec)

        train_losses.append(log_loss(y_train, train_prob, labels=classes))
        val_losses.append(log_loss(y_val, val_prob, labels=classes))

        y_pred = np.argmax(val_prob, axis=1)
        val_acc.append(accuracy_score(y_val, y_pred))
        val_f1.append(f1_score(y_val, y_pred, average="macro"))

    return train_losses, val_losses, val_acc, val_f1

results = {}
for name, eta in ETA_MAP.items():
    results[name] = run_sgd_experiment(eta)

{key: (results[key][2][-1], results[key][3][-1]) for key in results}  # (val acc, val f1)


In [ ]:
# Validation loss curves — all three presets
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: validation loss
ax = axes[0]
for name in ["TOO_LOW", "JUST_RIGHT", "TOO_HIGH"]:
    train_losses, val_losses, _, _ = results[name]
    ax.plot(range(1, EPOCHS+1), val_losses, marker="o", label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Validation loss vs epoch")
ax.legend()

# Right: train vs val for JUST_RIGHT only — shows generalisation gap
ax2 = axes[1]
train_losses, val_losses, val_acc, val_f1 = results["JUST_RIGHT"]
ax2.plot(range(1, EPOCHS+1), train_losses, marker="o", label="Train loss (JUST_RIGHT)")
ax2.plot(range(1, EPOCHS+1), val_losses, marker="o", label="Val loss (JUST_RIGHT)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Generalisation gap — JUST_RIGHT")
ax2.legend()

plt.tight_layout()
plt.show()

# Print final metrics
print(f"{'Preset':<12} {'Val Acc':>8} {'Val F1':>8}")
print("-" * 30)
for name in ["TOO_LOW", "JUST_RIGHT", "TOO_HIGH"]:
    _, _, acc, f1 = results[name]
    print(f"{name:<12} {acc[-1]:>8.4f} {f1[-1]:>8.4f}")


## Reading the curves — what is actually happening

**TOO_LOW (flat near 1.10):**
Gradient steps are so small (0.0001) that the weights barely move between epochs.
The model stays near its random starting point — validation loss hovers at log(3) ≈ 1.099,
which is exactly what a 3-class random guesser produces.

**JUST_RIGHT (steadily decreasing):**
The model is learning the signal through the noise. Notice it cannot reach 0 loss —
the 25% label noise sets a floor. No model can perfectly fit data where a quarter of
the labels are wrong. But it is finding the best it can do, steadily.

**TOO_HIGH (actively increasing):**
This is the one that surprises people. The model is not just unstable — it is getting
*worse* each epoch. Why?

With `eta0=5.0`, each update massively overshoots the optimum. The step is so large that
the weights fly past the minimum and land further away than they started. In clean data
this causes oscillation. With label noise it is worse: the model starts *memorising the
corrupted labels* — it learns which examples have wrong labels and builds a systematic
bias around them. Each epoch reinforces this, making validation performance progressively worse.

This is overfitting in its most visible form: training loss may be decreasing (the model
is fitting its data) while validation loss increases (it is fitting *noise*, not signal).

---

## Debrief questions (record answers in your worksheet Part C)

1. Which preset shows **underfitting**? What is the mathematical cause?
2. Which shows **overfitting** — and why does the noise amplify it?
3. What does the generalisation gap look like for `JUST_RIGHT`? Is it healthy?
4. How does the floor on `JUST_RIGHT` connect to the data quality discussion in Activity 6?
5. If you had to explain the `TOO_HIGH` curve to a stakeholder in two sentences, what would you say?
